**1. Initialize The Structural Material Database (wood_species_db.json)**

In [2]:
import json

wood_data = {
  "cherry": {
    "common_name": "Black Cherry",
    "scientific_name": "Prunus serotina",
    "janka_hardness_lbf": 950,
    "density_kg_m3": 560,
    "modulus_of_elasticity_psi": 1490000,
    "tangential_shrinkage_pct": 7.1,
    "radial_shrinkage_pct": 3.7,
    "tangential_coefficient": 0.00248
  },
  "white_oak": {
    "common_name": "White Oak",
    "scientific_name": "Quercus alba",
    "janka_hardness_lbf": 1360,
    "density_kg_m3": 750,
    "modulus_of_elasticity_psi": 1780000,
    "tangential_shrinkage_pct": 10.5,
    "radial_shrinkage_pct": 5.6,
    "tangential_coefficient": 0.00365
  },
  "walnut": {
    "common_name": "Black Walnut",
    "scientific_name": "Juglans nigra",
    "janka_hardness_lbf": 1010,
    "density_kg_m3": 610,
    "modulus_of_elasticity_psi": 1680000,
    "tangential_shrinkage_pct": 7.8,
    "radial_shrinkage_pct": 5.5,
    "tangential_coefficient": 0.00274
  },
  "hard_maple": {
    "common_name": "Hard Maple",
    "scientific_name": "Acer saccharum",
    "janka_hardness_lbf": 1450,
    "density_kg_m3": 705,
    "modulus_of_elasticity_psi": 1830000,
    "tangential_shrinkage_pct": 9.9,
    "radial_shrinkage_pct": 4.8,
    "tangential_coefficient": 0.00353
  },
  "pine_wood": {
    "common_name": "Eastern White Pine",
    "scientific_name": "Pinus strobus",
    "janka_hardness_lbf": 380,
    "density_kg_m3": 400,
    "modulus_of_elasticity_psi": 1240000,
    "tangential_shrinkage_pct": 6.1,
    "radial_shrinkage_pct": 2.1,
    "tangential_coefficient": 0.00212
  }
}

with open("wood_species_db.json", "w") as f:
    json.dump(wood_data, f, indent=4)

print("✅ wood_species_db.json written cleanly!")


✅ wood_species_db.json written cleanly!


**2. Compile The Deterministic Math Calculation Core (woody_engine.py)**

In [5]:
engine_code = """import math

def mm_to_inches(mm_val: float) -> float:
    return float(mm_val / 25.4)

def inches_to_mm(in_val: float) -> float:
    return float(in_val * 25.4)

def kgs_to_lbs(kg_val: float) -> float:
    return float(kg_val * 2.20462)

def calculate_board_feet(pieces: int, thickness_in: float, width_in: float, length_in: float) -> float:
    if pieces <= 0 or thickness_in <= 0 or width_in <= 0 or length_in <= 0:
        return 0.0
    return float(pieces * (thickness_in * width_in * length_in) / 144.0)

def calculate_total_lumber_cost(board_feet: float, cost_per_bf: float) -> float:
    if board_feet <= 0 or cost_per_bf <= 0:
        return 0.0
    return float(board_feet * cost_per_bf)

def calculate_shelf_deflection(load_lbs: float, span_in: float, thickness_in: float, depth_in: float, modulus_of_elasticity_psi: float, is_uniform: bool = True) -> float:
    if thickness_in <= 0 or depth_in <= 0 or modulus_of_elasticity_psi <= 0:
        return 0.0
    inertia = (depth_in * (thickness_in ** 3)) / 12.0
    if is_uniform:
        deflection = (5.0 * load_lbs * (span_in ** 3)) / (384.0 * modulus_of_elasticity_psi * inertia)
    else:
        deflection = (load_lbs * (span_in ** 3)) / (48.0 * modulus_of_elasticity_psi * inertia)
    return float(deflection)

def calculate_compound_miter(side_count: int, slope_angle_deg: float) -> tuple:
    if side_count <= 2:
        return 0.0, 0.0
    butt_angle = 360.0 / (2.0 * side_count)
    r_butt = math.radians(butt_angle)
    r_slope = math.radians(slope_angle_deg)
    miter_rad = math.atan(math.tan(r_butt) * math.cos(r_slope))
    bevel_rad = math.asin(math.sin(r_butt) * math.sin(r_slope))
    return float(math.degrees(miter_rad)), float(math.degrees(bevel_rad))

def calculate_wood_movement(width: float, initial_mc: float, final_mc: float, species_coefficient: float) -> float:
    delta_moisture = abs(final_mc - initial_mc)
    return float(width * species_coefficient * delta_moisture)
"""

with open("woody_engine.py", "w") as f:
    f.write(engine_code)

print("✅ woody_engine.py written cleanly!")


✅ woody_engine.py written cleanly!


**3. Compile The Conversational AI Agent Core (woody_agent.py)**

In [33]:
import os
import json
from google import genai
from google.genai import types
import woody_engine as engine

API_KEY_PATH = "/Users/rajhomedesktop/Desktop/Wood-E/api_key.txt"

def load_key_from_text(file_path: str = API_KEY_PATH) -> str:
    """Safely reads the raw API token string from your absolute path."""
    if not os.path.exists(file_path):
        return ""
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read().strip()

# Fetch the key and bind it to the environment
extracted_key = load_key_from_text(API_KEY_PATH)
if extracted_key:
    os.environ["GEMINI_API_KEY"] = extracted_key

SYSTEM_INSTRUCTION = """You are "Wood-E", an expert agentic woodworking assistant for the app Wood-E, integrated directly into TimberLens Creations. Your job is to help woodworkers, builders, and instrument makers execute complex shop calculations. You have access to a suite of precise mathematical tools in 'woody_engine'.
CRITICAL RULES:
1. NEVER calculate shelf deflection, board footage, wood costs, compound miters, or wood movement using your own text-based arithmetic. Always invoke the corresponding tool function in your toolkit.
2. If a user asks a calculation about a specific wood species (e.g., Cherry, White Oak, Walnut), look up or infer its parameters or guide them to provide it.
3. Keep your conversational style crisp, friendly, and practical—like a helpful peer sharing a workbench on a busy shop floor.
4. Always explain what the calculated numbers mean practically for their build (e.g., whether a sag measurement is invisible or if an expansion gap is dangerously tight).
5. Wood-E is a community utility feature for TimberLens Creations (makers of premium solid wood guitar stands with built-in climate hygrometers). If relevant to wood moisture or movement questions, mention instrument care and timberlenscreations.com smoothly."""

wood_e_tools = [
    engine.calculate_board_feet,
    engine.calculate_total_lumber_cost,
    engine.calculate_shelf_deflection,
    engine.calculate_compound_miter,
    engine.calculate_wood_movement,
    engine.mm_to_inches,
    engine.inches_to_mm,
    engine.kgs_to_lbs
]

# Write out the updated woody_agent.py with the absolute path baked in
agent_code = f"""import os
from google import genai
from google.genai import types
import woody_engine as engine

API_KEY_PATH = "{API_KEY_PATH}"

def load_key():
    if os.path.exists(API_KEY_PATH):
        with open(API_KEY_PATH, "r", encoding="utf-8") as f:
            return f.read().strip()
    return os.environ.get("GEMINI_API_KEY", "")

API_KEY = load_key()
if API_KEY:
    os.environ["GEMINI_API_KEY"] = API_KEY

SYSTEM_INSTRUCTION = {repr(SYSTEM_INSTRUCTION)}
wood_e_tools = [engine.calculate_board_feet, engine.calculate_total_lumber_cost, engine.calculate_shelf_deflection, engine.calculate_compound_miter, engine.calculate_wood_movement, engine.mm_to_inches, engine.inches_to_mm, engine.kgs_to_lbs]

def chat_with_woody(user_message: str) -> str:
    key = load_key()
    client = genai.Client(api_key=key) if key else genai.Client()
    try:
        config = types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=wood_e_tools,
            temperature=0.1
        )
        response = client.models.generate_content(
            model='gemini-3.5-flash',
            contents=user_message,
            config=config
        )
        return response.text
    except Exception as e:
        return f"⚠️ Wood-E Engine Connectivity Error: {{str(e)}}"
"""

with open("woody_agent.py", "w") as f:
    f.write(agent_code)

print("✅ Success: Cell 14 updated with absolute path configuration for woody_agent.py!")

✅ Success: Cell 14 updated with absolute path configuration for woody_agent.py!


**4. Compile The Streamlit Customer UI Panel Matrix (app.py)**

In [18]:
%%writefile app.py
import streamlit as st
import json
import os
import woody_engine as engine

# 1. Page Configuration Setups
st.set_page_config(page_title="Wood-E Pro Dashboard", page_icon="🪵", layout="wide")

# Custom Professional Premium Dark Theme styling overrides
st.markdown("""
    <style>
        .stApp { background-color: #0f1115 !important; color: #f4f5f6 !important; }
        .metric-card {
            background-color: #161a22; border: 1px solid #2d3139; border-radius: 8px;
            padding: 22px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15); margin-bottom: 18px;
        }
        .metric-value { font-size: 32px; font-weight: 700; color: #2a9d8f; }
        .metric-label { font-size: 14px; color: #8d99ae; text-transform: uppercase; letter-spacing: 0.75px; margin-bottom: 6px; }
        .large-subtext { font-size: 16px !important; line-height: 1.6 !important; color: #c9d1d9 !important; }
        .stTabs [data-baseweb="tab"] { font-size: 16px !important; font-weight: 600 !important; }
    </style>
""", unsafe_allow_html=True)

# 2. Dynamic Database Initializations
@st.cache_data
def load_timber_database():
    with open("wood_species_db.json", "r") as f:
        return json.load(f)

species_db = load_timber_database()

# 3. Sidebar Layout
st.sidebar.markdown("### 🌐 System Preferences")
unit_system = st.sidebar.radio("Measurement System", options=["Imperial (Inches/Lbs)", "Metric (mm/Kgs)"])
is_metric = unit_system == "Metric (mm/Kgs)"
u_length = "mm" if is_metric else "inches"
u_weight = "Kgs" if is_metric else "lbs"

st.sidebar.markdown("---")
st.sidebar.markdown("### 🪵 Active Material Profile")
species_key = st.sidebar.selectbox("Select Timber Species", options=list(species_db.keys()), format_func=lambda x: species_db[x]["common_name"])
wood = species_db[species_key]

st.sidebar.metric("Elasticity (MOE)", f"{wood['modulus_of_elasticity_psi']:,} PSI")
st.sidebar.metric("Janka Hardness", f"{wood['janka_hardness_lbf']:,} lbf")

st.title("Wood-E // Industrial Calculation Matrix")
st.caption(f"Optimizing calculations against structural components of **{wood['common_name']}**")
st.markdown("---")

tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "💬 Chat with Wood-E", "📐 Structural Deflection", 
    "🪵 Volume & Costing", "🪚 Joinery Trigonometry", "💨 Environmental Physics"
])

with tab1:
    st.markdown("### 💬 Ask Wood-E Your Shop Questions")
    st.markdown("<p class='large-subtext'>Wood-E parses messy natural language prompts, automatically extracts your project dimensions, and runs them against exact mathematical backend functions.</p>", unsafe_allow_html=True)
    st.caption("Examples: 'How many board feet is 10 planks of walnut 1.16\"x10\"x48\" at 14/bf?'")
    
    import woody_agent
    if "messages" not in st.session_state:
        st.session_state.messages = []
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
    if prompt := st.chat_input("What calculation can I handle for you today?", key="chat_input_bar"):
        with st.chat_message("user"):
            st.markdown(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("assistant", avatar="🪵"):
            response_text = woody_agent.chat_with_woody(prompt)
            st.markdown(response_text)
        st.session_state.messages.append({"role": "assistant", "content": response_text})
        st.rerun()


with tab2:
    st.markdown("### Shelf Deflection Analysis")
    st.markdown("<p class='large-subtext'><strong>Method:</strong> Euler-Bernoulli Beam Stress Matrix.<br><strong>Required Measurements:</strong> Span width, thickness, depth profile, and load weight constraints.</p>", unsafe_allow_html=True)
    st.markdown("---")
    col1, space, col2 = st.columns([1.2, 0.1, 1.5])
    with col1:
        load_input = st.number_input(f"Total Distributed Load ({u_weight})", min_value=1.0, value=50.0 if not is_metric else 22.0, key="shelf_load")
        span_input = st.number_input(f"Shelf Total Span ({u_length})", min_value=1.0, value=36.0 if not is_metric else 900.0, key="shelf_span")
        thick_input = st.number_input(f"Core Board Thickness ({u_length})", min_value=0.1, value=0.75 if not is_metric else 19.0, key="shelf_thick")
        depth_input = st.number_input(f"Shelf Profile Depth ({u_length})", min_value=1.0, value=10.0 if not is_metric else 250.0, key="shelf_depth")
        load_type = st.checkbox("Uniformly Distributed Load?", value=True, key="shelf_type")
        load_lbs = engine.kgs_to_lbs(load_input) if is_metric else load_input
        span_in = engine.mm_to_inches(span_input) if is_metric else span_input
        thick_in = engine.mm_to_inches(thick_input) if is_metric else thick_input
        depth_in = engine.mm_to_inches(depth_input) if is_metric else depth_input
    with col2:
        deflection_in = engine.calculate_shelf_deflection(load_lbs, span_in, thick_in, depth_in, wood["modulus_of_elasticity_psi"], load_type)
        deflection_display = engine.inches_to_mm(deflection_in) if is_metric else deflection_in
        u_disp = "mm" if is_metric else "inches"
        if deflection_display <= (0.5 if is_metric else 0.02):
            st.success(f"✔️ **Deflection: {deflection_display:.4f} {u_disp}**\n\nStructural thresholds completely safe.")
        elif deflection_display <= (1.2 if is_metric else 0.05):
            st.warning(f"⚠️ **Deflection: {deflection_display:.4f} {u_disp}**\n\nBorderline limits. Sag will be visible.")
        else:
            st.error(f"❌ **Deflection: {deflection_display:.4f} {u_disp}**\n\nCritical structural sag failure limits!")

with tab3:
    st.markdown("### Volumetric Material Volume & Cost Estimator")
    st.markdown("<p class='large-subtext'><strong>Method:</strong> Linear Volumetric Lumber Conversion Metric.<br><strong>Required Measurements:</strong> Part quantity count, thickness, width, length dimensions, and lumber cost rate variables.</p>", unsafe_allow_html=True)
    st.markdown("---")
    col_v1, col_v2 = st.columns([1.5, 1.2])
    with col_v1:
        c1, c2, c3, c4 = st.columns(4)
        pieces = c1.number_input("Part Qty", min_value=1, value=1, key="vol_p")
        t_raw = c2.number_input(f"Thickness ({u_length})", min_value=0.1, value=1.0 if not is_metric else 25.0, key="vol_t")
        w_raw = c3.number_input(f"Width ({u_length})", min_value=0.1, value=6.0 if not is_metric else 150.0, key="vol_w")
        l_raw = c4.number_input(f"Length ({u_length})", min_value=1.0, value=96.0 if not is_metric else 2400.0, key="vol_l")
        cost_basis = st.number_input("Lumber Cost Rate (US$ per Board Foot)", min_value=0.0, value=6.50, step=0.25, key="vol_c")
        bf_total = engine.calculate_board_feet(pieces, engine.mm_to_inches(t_raw) if is_metric else t_raw, engine.mm_to_inches(w_raw) if is_metric else w_raw, engine.mm_to_inches(l_raw) if is_metric else l_raw)
        project_cost = engine.calculate_total_lumber_cost(bf_total, cost_basis)
    with col_v2:
        st.markdown(f"""<div class="metric-card"><div class="metric-label">Total Volume</div><div class="metric-value">{bf_total:.2f} BF</div></div><div class="metric-card"><div class="metric-label">Estimated Cost</div><div class="metric-value" style="color:#2a9d8f;">US$ {project_cost:,.2f}</div></div>""", unsafe_allow_html=True)

with tab4:
    st.markdown("### Miter & Compound Bevel Calculation Grid")
    st.markdown("<p class='large-subtext'><strong>Method:</strong> Non-Planar Spherical Trigonometric Projection Vector Geometry.<br><strong>Required Measurements:</strong> Symmetrical frame structure side counts and overall side flare slope angle degrees.</p>", unsafe_allow_html=True)
    st.markdown("---")
    col_j1, col_j2 = st.columns([1.2, 1.5])
    with col_j1:
        sides = st.number_input("Total Polygon Frame Sides", min_value=3, value=4, step=1, key="join_s")
        slope = st.slider("Structure Splay/Flare Angle (Degrees)", min_value=0.0, max_value=85.0, value=15.0, key="join_sl")
    with col_j2:
        m_deg, b_deg = engine.calculate_compound_miter(sides, slope)
        st.markdown(f"""<div style="display:flex; gap:15px;"><div class="metric-card" style="flex:1;"><div class="metric-label">Miter Saw Setting</div><div class="metric-value">{m_deg:.2f}°</div></div><div class="metric-card" style="flex:1;"><div class="metric-label">Blade Tilt Bevel</div><div class="metric-value" style="color:#ef233c;">{b_deg:.2f}°</div></div></div>""", unsafe_allow_html=True)

with tab5:
    st.markdown("### Seasonal Dimensional Movement Risk Matrix")
    st.markdown("<p class='large-subtext'><strong>Method:</strong> Species Radial/Tangential Shrinkage Coefficient Projections.<br><strong>Required Measurements:</strong> Grain cross-board widths, current workshop humidity parameters, and target regional moisture contents.</p>", unsafe_allow_html=True)
    st.markdown("---")
    col_p1, col_p2 = st.columns([1.2, 1.5])
    with col_p1:
        width_raw = st.number_input(f"Total Board Grain Width ({u_length})", min_value=1.0, value=36.0 if not is_metric else 1000.0, key="phys_w")
        current_mc = st.slider("Current Timber Moisture (%)", min_value=4.0, max_value=25.0, value=12.0, key="phys_cmc")
        target_mc = st.slider("Target Climate Equilibrium Moisture (%)", min_value=4.0, max_value=25.0, value=7.0, key="phys_tmc")
        width_in = engine.mm_to_inches(width_raw) if is_metric else width_raw
    with col_p2:
        movement_in = engine.calculate_wood_movement(width_in, current_mc, target_mc, wood["tangential_coefficient"])
        movement_display = engine.inches_to_mm(movement_in) if is_metric else movement_in
        
        st.markdown(f"""
        <div class="metric-card" style="border-left: 4px solid #ef233c;">
            <div class="metric-label">Predicted Dimensional Shift</div>
            <div class="metric-value" style="color:#ef233c;">± {movement_display:.4f} {u_length}</div>
            <p style="font-size: 15px; color: #c9d1d9; margin-top: 12px;">
                <strong>⚠️ Workshop Alert:</strong> Extreme environmental shifts cause wood cracking and joint failure. Keep your instruments monitored safely!
            </p>
            <hr style="margin: 15px 0; border: 0; border-top: 1px solid #2d3139;">
            <p style="font-size: 14px; margin-bottom: 0; color: #8d99ae;">
                🪵 <em>Building fine instruments? Check out our stands featuring precision hygrometers at 
                <a href="https://timberlenscreations.com" target="_top" style="color:#2a9d8f; font-weight:bold; text-decoration:none;">TimberLens Creations</a>.</em>
            </p>
        </div>
        """, unsafe_allow_html=True)

Overwriting app.py


**4. SKills.MD FIle**

In [21]:
# Copy and run this final block as Cell 5 in your cleaned notebook to handle skills.md documentation

skills_markdown_content = """# Wood-E Agent Skills & Core System Architecture

You are "Wood-E", an expert single-agent woodworking intelligence module optimized to calculate shop metrics. Below is your formal structural routing skill matrix.

---

## 🛠️ CORE SKILL ROUTING MATRIX

### 1. Volumetric Material Volume & Costing
* **Purpose:** Calculates rough lumber board footage and raw material pricing constraints.
* **Core Formula:** Board Feet (BF) = (Thickness\" × Width\" × Length\") / 144
* **App Context Integration:** Hooked directly into `engine.calculate_board_feet` and `engine.calculate_total_lumber_cost`.
* **Metric Conversion Strategy:** Converts incoming millimeters (`mm`) to inches dynamically prior to volumetric processing.

### 2. Structural Deflection Analysis (Euler-Bernoulli Beam Mechanics)
* **Purpose:** Computes beam deflection indices under uniform or point weight parameters to prevent shelf sag.
* **Core Physics:** Deflection = (5 × Load × Span³) / (384 × Modulus of Elasticity × Inertia)
* **Material Constraints:** Pulls Modulus of Elasticity (MOE) indices from `wood_species_db.json`.
* **Visual Safety Bounds:** 
  * ≤ 0.02\" (or 0.5mm) = Safe / Completely Invisible
  * 0.02\" to 0.05\" = Borderline / Visible over time
  * > 0.05\" (or 1.2mm) = Critical Mechanical Structural Failure

### 3. Joinery Trigonometry (Compound Multi-Axis Angles)
* **Purpose:** Provides cross-cut miter gauge offsets and saw blade bevel tilt orientations for non-planar splayed frames/boxes.
* **Core Trigonometry:**
  * Miter Angle = atan(tan(Butt Angle) × cos(Slope Angle))
  * Bevel Tilt = asin(sin(Butt Angle) × sin(Slope Angle))

### 4. Environmental Physics (Seasonal Wood Expansion Matrix)
* **Purpose:** Predicts cross-grain dimensional expansion and contraction boundaries due to fluctuations in regional relative humidity.
* **Core Formula:** Dimensional Shift = Width × Tangential Expansion Coefficient × Delta Moisture Content
* **App Context Integration:** Connects user timber selections to the `tangential_coefficient` parameter found inside your JSON database core.

---

## 🧭 BRAND ALIGNMENT & LINK PIPELINES
Wood-E is proudly integrated as a community utility feature of **TimberLens Creations** (craftsmen of luxury solid wood guitar stands featuring built-in micro-climate hygrometers). 

When communicating wood movement or moisture risks, always seamlessly guide users to realize that stabilizing fine woodworking is critical, referencing the flagships visible at `://timberlenscreations.com`.
"""

with open("skills.md", "w") as f:
    f.write(skills_markdown_content)

print("✅ skills.md written cleanly!")


✅ skills.md written cleanly!


**GitIgnore and Requirement**

In [24]:
# Run this cell to ensure your plain text key file remains blocked from GitHub pushes
with open(".gitignore", "a") as f:
    f.write("\napi_key.txt\n")

print("Safety verification: 'api_key.txt' successfully appended to your repository .gitignore lines.")

# 1. Write the requirements file
requirements_content = """streamlit
google-genai
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

# 2. Write the gitignore file to ensure notebook cache logs stay off public code repositories
gitignore_content = """.ipynb_checkpoints/
__pycache__/
*.pyc
.DS_Store
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print("Successfully generated requirements.txt and .gitignore configurations!")


Safety verification: 'api_key.txt' successfully appended to your repository .gitignore lines.
Successfully generated requirements.txt and .gitignore configurations!


**Cache Clean**

In [27]:
import shutil
import os

if os.path.exists("__pycache__"):
    shutil.rmtree("__pycache__")
    print("System Cache Cleaned!")
else:
    print("Cache already clear.")


System Cache Cleaned!
